In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# disable warnings
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append(r'.')
import utils
import importlib
importlib.reload(sys.modules['utils'])
from utils import *

In [ ]:
# regression

import stata_setup

stata_setup.config("C:/Program Files/Stata17/", "se")
from pystata import stata

In [3]:
data = pd.read_csv("data1.csv")
data5 = pd.read_csv("data5.csv").rename(columns={"n": "year_n"})

In [4]:
r = 10

df_norm = (data.merge(data5, on=["PersonId", 'DegreeYear', 'phd_community'], how="inner")
    # get annee biblio
    .assign(Annee_Bibliographique=lambda x: x["tau"] + x["DegreeYear"])
    .fillna({"collab_num": 0, "cite_norm": 0})
    .drop_duplicates()
)

df_norm

,PersonId,phd_community,phd_area,prof_community,prof_area,DegreeYear,rao,gender,Percentile,DegreePercentile,...,rank20,degree_rank5,degree_rank10,degree_rank15,degree_rank20,is_cross,is_far,tau,year_n,Annee_Bibliographique
0,1,4,Sociology,4,Sociology,2008,0.527476,M,92.070485,97.356828,...,1.0,1.0,1.0,1.0,1.0,0,0,-4,1.0,2004
1,1,4,Sociology,4,Sociology,2008,0.527476,M,92.070485,97.356828,...,1.0,1.0,1.0,1.0,1.0,0,0,-3,0.0,2005
2,1,4,Sociology,4,Sociology,2008,0.527476,M,92.070485,97.356828,...,1.0,1.0,1.0,1.0,1.0,0,0,-2,1.0,2006
3,1,4,Sociology,4,Sociology,2008,0.527476,M,92.070485,97.356828,...,1.0,1.0,1.0,1.0,1.0,0,0,-1,0.0,2007
4,1,4,Sociology,4,Sociology,2008,0.527476,M,92.070485,97.356828,...,1.0,1.0,1.0,1.0,1.0,0,0,0,1.0,2008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475945,31775,3,Medicine,3,Health,2015,0.436724,M,68.825911,68.072289,...,0.0,0.0,0.0,0.0,0.0,1,0,2,0.0,2017
475946,31775,3,Medicine,3,Health,2015,0.436724,M,68.825911,68.072289,...,0.0,0.0,0.0,0.0,0.0,1,0,3,1.0,2018
475947,31775,3,Medicine,3,Health,2015,0.436724,M,68.825911,68.072289,...,0.0,0.0,0.0,0.0,0.0,1,0,4,1.0,2019
475948,31775,3,Medicine,3,Health,2015,0.436724,M,68.825911,68.072289,...,0.0,0.0,0.0,0.0,0.0,1,0,5,0.0,2020


In [ ]:
# analysis period
tau_range = np.arange(-4, 11)

t = df_norm.copy()

ts = []

for i, area in enumerate(list(community_names.keys())):
    t_area = t.query("phd_community == @area").query("gender in ['M', 'F'] and adv_gender in ['M', 'F']")
    t_area['tau'] = t_area['tau'] + 5
    
    # # standardize
    t_person = t_area[['PersonId', 'rao']].drop_duplicates()
    t_person['rao'] = (t_person['rao'] - t_person['rao'].mean()) / t_person['rao'].std()
    t_area = t_area.drop(columns="rao").merge(t_person, on='PersonId', how='inner')
    
    print(f"Area: {area}")

    command = f"""
    encode(gender), gen(gender2)
    encode(adv_gender), gen(adv_gender2)
    qui poisson year_n i.tau##i.rank{r}##c.rao DegreePercentile cite_norm collab_num gender2 adv_gender2 adv_n seniority i.Annee_Bibliographique
    margins tau, dydx(rao) at(rank{r}=(0 1))
    """
    
    #  i.tau c.rao i.top i.tau#c.rao i.tau#c.rao#i.top
    
    stata.run("clear")
    stata.pdataframe_to_data(t_area)
    stata.run(command)
    
    # collect results
    results = stata.get_return()["r(table)"].T
    results = pd.DataFrame(results).iloc[:, :6]
    results.columns = ["coef", "se", "z", "p", "ci_low", "ci_high"]
    results[f"is_top"] = [0] * len(tau_range) + [1] * len(tau_range)
    results['tau'] = [i for i in tau_range] * 2
    results["phd_community"] = area
    ts.append(results)
    
data = pd.concat(ts)

In [18]:
# plot
groupvar, group = "phd_community", list(community_names.keys())
titles = [name for name in community_names.values()]

fig = make_subplots(
    rows=1,
    cols=5,
    subplot_titles=titles,
    vertical_spacing=0.15,
    horizontal_spacing=0.06,
)
names = ["Non-top", f"Top {r}%"]
colors = generate_gradient_colors(2)

for i, domain in enumerate(group):
    for j, top in enumerate(range(2)):

        t1_domain = data.query(f"{groupvar} == @domain and is_top == @top")
        t1 = t1_domain.dropna()

        x = tau_range
        y1 = t1["coef"].to_numpy()
        y1_upper, y1_lower = (
            t1["ci_low"].to_numpy(),
            t1["ci_high"].to_numpy(),
        )

        fig = add_lines_with_errorband(
            fig,
            x,
            y1,
            y1_upper,
            y1_lower,
            name=names[j],
            color=colors[j],
            line_alpha=0.85,
            band_alpha=0.14,
            showlegend=True if i == 0 else False,
            row=1,
            col=i % 5 + 1,
        )

# Update layout for subplot spacing and axes titles
fig.update_layout(
    width=780,
    height=210,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=10, color="rgba(30,30,30,1)"),
    margin=dict(l=52, r=32, t=34, b=56),
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    gridwidth=0.5,
    showline=True,
    linecolor="rgba(60,60,60,0.55)",
    linewidth=0.8,
    ticks="outside",
    ticklen=3,
    tickwidth=0.8,
    tickcolor="rgba(60,60,60,0.55)",
    tickfont=dict(size=9),
    zeroline=False,
)
fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    gridwidth=0.5,
    showline=True,
    linecolor="rgba(60,60,60,0.55)",
    linewidth=0.8,
    tickfont=dict(size=9),
    zeroline=False,
)
# update legend title
fig.update_layout(
    legend=dict(
        title="Faculty university rank",
        orientation="h",
        yanchor="top",
        y=-0.3,
        yref="paper",
        xanchor="center",
        x=0.5,
        xref="paper",
    )
)
fig.update_yaxes(
    title=dict(text="Productivity change<br>per 1 SD increase<br>in PhD interdisciplinarity"),
    tickfont=dict(color="black"),
    col=1,
)

# zeroline
fig.add_vline(x=0, line=dict(dash="dash", width=1))
fig.add_hline(y=0, line=dict(dash="dash", width=1))

# Update layout to include super x and y labels
fig.update_xaxes(title=dict(text=None, standoff=0))

for ann in fig.layout.annotations:
    ann.font = dict(size=13, color="black")
    ann.y = 1.02

fig.add_annotation(
    text="Relative year to PhD graduation",
    x=0.5,
    y=-0.2,
    showarrow=False,
    xref="paper",
    xanchor="center",
    yref="paper",
    yanchor="top",
    font=dict(size=10),
)

fig.show()
fig.write_image("fig5a.svg", width=780, height=210)

In [32]:
# get mean, std
t = df_norm[['PersonId', 'rao', 'phd_community']].drop_duplicates().groupby(['phd_community']).agg(
    mean_rao=('rao', 'mean'),
    std_rao=('rao', 'std')
).reset_index()

# merge with data
data = df_norm.merge(t, on='phd_community', how='left')

def group_rao(row):
    if row['rao'] < row['mean_rao']:
        return 0
    return 1
data['rao_cut'] = data.apply(group_rao, axis=1)

df_cite_norm = (data
    [['phd_community','PersonId', 'rao_cut', 'rank10', 'tau', 'year_n']]
    .drop_duplicates()
    .dropna()
    .groupby(['phd_community', 'rao_cut', 'rank10'])
    .apply(lambda x: apply_lowess(x, n_boot=100, xcol='tau', ycol='year_n'))
    .reset_index()
)

df_cite_norm

Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping
Using 16 cores for bootstrapping


,phd_community,rao_cut,rank10,level_3,tau,year_n,lower,upper
0,0,0,0.0,0,-4.0,0.112727,0.080451,0.170320
1,0,0,0.0,1224,-3.0,0.265249,0.251809,0.314367
2,0,0,0.0,2448,-2.0,0.422463,0.415025,0.465305
3,0,0,0.0,3672,-1.0,0.585208,0.570537,0.625879
4,0,0,0.0,4896,0.0,0.758520,0.719329,0.778971
...,...,...,...,...,...,...,...,...
295,4,1,1.0,7756,6.0,1.211510,1.179853,1.255952
296,4,1,1.0,8408,7.0,1.206045,1.187703,1.250856
297,4,1,1.0,9008,8.0,1.199636,1.185493,1.262354
298,4,1,1.0,9550,9.0,1.192335,1.175010,1.274024


In [38]:
# plot
rao_colors = ['rgb(0, 114, 188)', 'rgb(211, 45, 45)']
rao_labels = ['Below mean', 'Above mean']

subplot_titles=list(community_names.values())
subplot_titles[2] =  r"<b>Top 10% universities</b><br>" + subplot_titles[2]
subplot_titles =  subplot_titles + ['', '', '<b>Other universities</b>', '', '']

fig = make_subplots(
    rows=2,
    cols=5,
    shared_yaxes=False,
    subplot_titles=subplot_titles,
)

for i, area in enumerate(list(community_names.keys())):
    for j, rao_cut in enumerate([0, 1]):
        for k, istop in enumerate([True, False]):
            t = df_cite_norm.query("phd_community == @area and rao_cut == @rao_cut and rank10 == @istop")
            x = t["tau"]
            y = t["year_n"]
            upper, lower = t["upper"], t["lower"]
            add_lines_with_errorband(
                fig,
                x=x,
                y=y,
                upper=upper,
                lower=lower,
                name=rao_labels[j] if i == 4 and k==1 else "",
                color=rao_colors[j],
                showlegend=True if i == 4 and k==1 else False,
                showband=True,
                dash=None,
                row=int(k) + 1,
                col=i % 5 + 1,
            )

fig.update_layout(
    width=650,
    height=380,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Arial", size=10, color="rgba(30,30,30,1)"),
    margin=dict(l=88, r=24, t=42, b=62),
)

fig.update_xaxes(
    showgrid=False,
    showline=True,
    linecolor="rgba(60,60,60,0.55)",
    linewidth=0.8,
    ticks="outside",
    ticklen=3,
    tickwidth=0.8,
    tickcolor="rgba(60,60,60,0.55)",
    tickfont=dict(size=9),
    zeroline=False,
    dtick=2,
)
fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    gridwidth=0.5,
    showline=True,
    linecolor="rgba(60,60,60,0.55)",
    linewidth=0.8,
    ticks="outside",
    ticklen=3,
    tickwidth=0.8,
    tickcolor="rgba(60,60,60,0.55)",
    tickfont=dict(size=9),
    zeroline=False,
)

# legend title
fig.update_layout(
    legend=dict(
        title=dict(text="PhD interdisciplinarity"),
        orientation="h",
        yanchor="top",
        y=-0.2,
        yref="paper",
        xanchor="center",
        x=0.5,
        xref="paper",
    )
)
fig.update_yaxes(
    title=dict(text="Annual productivity"),
    tickfont=dict(color="black"),
    col=1,
)
fig.update_xaxes(title=dict(text=None, standoff=0))

for ann in fig.layout.annotations:
    ann.font = dict(size=12, color="black")

fig.add_annotation(
    text="Years since PhD graduation",
    x=0.5,
    y=-0.12,
    showarrow=False,
    xref="paper",
    xanchor="center",
    yref="paper",
    yanchor="top",
    font=dict(size=10),
)

# ref line
fig.add_vline(x=0, line=dict(color="rgba(60,60,60,0.55)", width=1, dash="dash"))

fig.show()
fig.write_image("fig5b.svg", width=650, height=380)